In [98]:
# Import the required libraries
import pandas as pd
# ==========================
# STEP 1: Load the Dataset
# ==========================

# Read the CSV file using cp1252 encoding because the dataset
# contains special French characters (é, è, ç, etc.)
data = pd.read_csv("vital_records_pipeline.csv", encoding="cp1252")


# ==========================
# STEP 2: Rename Columns
# ==========================

# Rename 'Primary_Name' to a more meaningful column name
data.rename(columns={"Primary_Name": "Full_Name"}, inplace=True)


# ==========================
# STEP 3: Clean Full_Name
# ==========================

# - Convert names to Title Case
# - Remove extra spaces between words
data["Full_Name"] = (
    data["Full_Name"]
        .str.title()
        .str.split()
        .str.join(" ")
)


# ==========================
# STEP 4: Standardize French Month Names
# ==========================

# Dictionary to convert French month names into English
month_map = {
    "Jan": "Jan",
    "Fév": "Feb",
    "Mar": "Mar",
    "Avr": "Apr",
    "Mai": "May",
    "Juin": "Jun",
    "Juil": "Jul",
    "Aoû": "Aug",
    "Sep": "Sep",
    "Oct": "Oct",
    "Nov": "Nov",
    "Déc": "Dec"
}

# Replace French month names with English
for french, english in month_map.items():
    data["Event_Date"] = data["Event_Date"].str.replace(
        french,
        english,
        regex=False
    )


# ==========================
# STEP 5: Convert Event_Date
# ==========================

# Convert mixed date formats into datetime format
# dayfirst=True because the dataset follows DD-MM-YYYY
# errors='coerce' converts invalid dates into NaT
data["Event_Date"] = pd.to_datetime(
    data["Event_Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)


# ==========================
# STEP 6: Standardize Event Types
# ==========================

# Convert multilingual event names into standard English
event_map = {

    # Death
    "Death": "Death",
    "Tod": "Death",
    "Décès": "Death",

    # Birth
    "Birth": "Birth",
    "Geburt": "Birth",
    "Naissance": "Birth",

    # Marriage
    "Marriage": "Marriage",
    "Heirat": "Marriage",
    "Mariage": "Marriage"
}

data["Event_Type"] = data["Event_Type"].replace(event_map)


# ==========================
# STEP 7: Clean City Names
# ==========================

# Remove leading/trailing spaces
# Convert city names into Title Case
data["City"] = data["City"].str.strip().str.title()


# ==========================
# STEP 8: Sort Records
# ==========================

# Arrange records in ascending order based on Record_ID
data = data.sort_values(by="Record_ID")


# ==========================
# STEP 9: Remove Duplicate Records
# ==========================

# Remove duplicate rows and reset the index
data = data.drop_duplicates(ignore_index=True)


# ==========================
# STEP 10: Save the Cleaned Dataset
# ==========================

# Export the cleaned dataset to a new CSV file
data.to_csv("cleaned_vital_records.csv", index=False)


# ==========================
# STEP 11: Display the Final Dataset
# ==========================

print(data)

     Record_ID Event_Type Event_Date         Full_Name       City
0    CERT-1000      Death 1991-12-16    Sophie Fischer      Paris
1    CERT-1001   Marriage 1995-08-11      Anna Fischer     Berlin
2    CERT-1002      Birth 1996-02-24      Klaus Dubois        NaN
3    CERT-1003   Marriage 1996-09-02       Jean Dubois       Lyon
4    CERT-1004      Death 1991-08-04     Julia Bernard       Lyon
..         ...        ...        ...               ...        ...
495  CERT-1495      Birth 1990-02-06        Klaus Roux  Marseille
496  CERT-1496      Death 1999-12-16         Jean Roux    Hamburg
497  CERT-1497      Birth 1999-01-13  Sophie Schneider    Hamburg
498  CERT-1498      Birth 1995-07-17         Jean Roux       Lyon
499  CERT-1499      Birth 1999-03-14   Klaus Schneider     Berlin

[500 rows x 5 columns]
